# LeetCode 3244 — Shortest Distance After Road Addition Queries II bằng Python

- Link: https://leetcode.com/problems/shortest-distance-after-road-addition-queries-ii/
- Độ khó: **Hard**
- Chủ đề: Khai thác điều kiện **các đường mới không cắt chéo nhau** ⇒ đường đi ngắn nhất luôn là "chuỗi nhảy" trên các thành phố **còn sống**; mỗi đường mới **xóa** các thành phố nằm giữa ⇒ danh sách liên kết `nxt[]` (hoặc DSU "phần tử kế tiếp còn sống"), tổng `O(n + q)`

Skeleton trên LeetCode (Python3):

```python
class Solution:
    def shortestDistanceAfterQueries(self, n: int, queries: List[List[int]]) -> List[int]:
```

## 1. Mục tiêu học

1. Đồ thị ban đầu là đường thẳng `0 → 1 → … → n−1`; mỗi truy vấn thêm cạnh **tiến** `u → v` (`u < v`). Đề bảo đảm không có hai đường `u_i < u_j < v_i < v_j` (không **cắt chéo**): hai đường hoặc rời nhau, hoặc lồng nhau.
2. Với các khoảng lồng nhau/rời nhau, đường ngắn nhất từ 0 tới `n−1` đi qua các **khoảng ngoài cùng**: dùng mọi đường "ngoài cùng", phần còn lại đi bộ `+1`.
3. Duy trì `nxt[i]` = thành phố kế tiếp trên đường ngắn nhất hiện tại. Thêm `u → v`: nếu `u` còn sống và `nxt[u] < v`, mọi thành phố sống trong `(u, v)` bị **loại** (mỗi thành phố bị loại tối đa một lần) ⇒ tổng `O(n + q)`.
4. Đáp án = số thành phố còn sống − 1.
5. Kiểm chứng bằng BFS sau mỗi truy vấn.

## 2. Đề bài & ví dụ

Có `n` thành phố `0..n−1`, ban đầu có đường một chiều `i → i+1`. Truy vấn `[u, v]` thêm đường một chiều `u → v`. Sau mỗi truy vấn, trả về độ dài đường đi ngắn nhất từ `0` tới `n−1`. Không có đường lặp lại; không có hai truy vấn `i ≠ j` với `u_i < u_j < v_i < v_j`.

**Ví dụ 1:** `n = 5`, `[[2,4],[0,2],[0,4]]` → `[3,2,1]`.
**Ví dụ 2:** `n = 4`, `[[0,3],[0,2]]` → `[1,1]`.

**Ràng buộc:** `3 ≤ n ≤ 10^5`; `1 ≤ q ≤ 10^5`; `0 ≤ u < v < n`, `1 < v − u`.

In [1]:
from typing import List
from collections import deque

examples = [(5, [[2, 4], [0, 2], [0, 4]], [3, 2, 1]), (4, [[0, 3], [0, 2]], [1, 1])]
for n, q, exp in examples:
    print('n =', n, q, '-> kỳ vọng', exp)

n = 5 [[2, 4], [0, 2], [0, 4]] -> kỳ vọng [3, 2, 1]
n = 4 [[0, 3], [0, 2]] -> kỳ vọng [1, 1]


## 3. Kiến thức nền

### 3.1. BFS trên đồ thị có hướng không trọng số

In [2]:
def bfs_dist(n, adj):
    d = [-1] * n; d[0] = 0; q = deque([0])
    while q:
        u = q.popleft()
        for v in adj[u]:
            if d[v] < 0:
                d[v] = d[u] + 1; q.append(v)
    return d[n - 1]

adj = [[i + 1] for i in range(4)] + [[]]
adj[2].append(4)
print('n = 5 với đường 2 -> 4:', bfs_dist(5, adj))

n = 5 với đường 2 -> 4: 3


### 3.2. Khoảng không cắt chéo

Hai khoảng `[u_i, v_i]`, `[u_j, v_j]` không cắt chéo ⇒ hoặc rời nhau (chỉ có thể chung đầu mút), hoặc một khoảng **chứa** khoảng kia. Khi đã có `u → v`, mọi đường nằm trong `[u, v]` trở nên vô dụng đối với đường đi ngắn nhất (nhảy thẳng `u → v` luôn tốt hơn hoặc bằng).

## 4. Ý tưởng nền tảng / cách nghĩ trực giác

Ví dụ 1 (`n = 5`):

- Thêm `2 → 4`: loại thành phố `3` ⇒ còn `0, 1, 2, 4` ⇒ `3`.
- Thêm `0 → 2`: loại `1` ⇒ còn `0, 2, 4` ⇒ `2`.
- Thêm `0 → 4`: loại `2` ⇒ còn `0, 4` ⇒ `1` ✓.

Ví dụ 2: `0 → 3` loại `1, 2`; `0 → 2` nằm trong khoảng đã có (`nxt[0] = 3 ≥ 2`) ⇒ không đổi.

## 5. Cách 1 — Brute Force: BFS sau mỗi truy vấn

In [3]:
def brute(n, queries):
    adj = [[i + 1] for i in range(n - 1)] + [[]]
    out = []
    for u, v in queries:
        adj[u].append(v)
        out.append(bfs_dist(n, adj))
    return out

for n, q, exp in examples:
    print(brute(n, q), '| kỳ vọng', exp)

[3, 2, 1] | kỳ vọng [3, 2, 1]
[1, 1] | kỳ vọng [1, 1]


**Độ phức tạp:** `O(q · (n + q))`.

## 6. Ý tưởng thuật toán / bước đột phá

```
nxt[i] = i + 1;  alive = n
for (u, v):
    if nxt[u] != −1 and nxt[u] < v:          # u còn sống và đường mới vươn xa hơn
        j = nxt[u]
        while j < v: k = nxt[j]; nxt[j] = −1; alive −= 1; j = k
        nxt[u] = v
    answer.append(alive − 1)
```

Mỗi thành phố bị đánh dấu `−1` tối đa một lần ⇒ tổng `O(n + q)`.

## 7. Các cách tiếp theo

### 7.1. Cách 2 — DP đường đi ngắn nhất theo thứ tự tô-pô (mọi cạnh tiến) sau mỗi truy vấn

In [4]:
def dp_recompute(n, queries):
    radj = [[] for _ in range(n)]           # radj[v] = các u có cạnh u -> v (ngoài i -> i+1)
    out = []
    for u, v in queries:
        radj[v].append(u)
        d = [0] * n
        for x in range(1, n):
            best = d[x - 1] + 1
            for w in radj[x]:
                if d[w] + 1 < best:
                    best = d[w] + 1
            d[x] = best
        out.append(d[n - 1])
    return out

for n, q, exp in examples:
    print(dp_recompute(n, q), '| kỳ vọng', exp)

[3, 2, 1] | kỳ vọng [3, 2, 1]
[1, 1] | kỳ vọng [1, 1]


Không dùng điều kiện không cắt chéo — đúng cho mọi đồ thị cạnh tiến, `O(q · (n + q))`.

### 7.2. Cách 3 — Danh sách liên kết `nxt[]` (lời giải chính)

In [5]:
def linked_list(n, queries):
    nxt = list(range(1, n + 1))
    alive = n
    out = []
    for u, v in queries:
        if nxt[u] != -1 and nxt[u] < v:
            j = nxt[u]
            while j < v:
                k = nxt[j]; nxt[j] = -1; alive -= 1; j = k
            nxt[u] = v
        out.append(alive - 1)
    return out

for n, q, exp in examples:
    print(linked_list(n, q), '| kỳ vọng', exp)

[3, 2, 1] | kỳ vọng [3, 2, 1]
[1, 1] | kỳ vọng [1, 1]


### 7.3. Cách 4 — DSU "thành phố còn sống kế tiếp"

`find(x)` = thành phố còn sống nhỏ nhất `≥ x`. Đường `u → v` hữu ích ⇔ `u` còn sống và có thành phố còn sống trong `(u, v)`; khi đó loại lần lượt `find(u + 1)` cho tới khi `≥ v` (nối `parent[x] = x + 1`). Nén đường đi giữ độ phức tạp gần tuyến tính.

In [6]:
def dsu_next_alive(n, queries):
    parent = list(range(n + 1))
    def find(x):
        root = x
        while parent[root] != root:
            root = parent[root]
        while parent[x] != root:
            parent[x], x = root, parent[x]
        return root
    alive = n
    out = []
    for u, v in queries:
        if find(u) == u:                     # u còn sống
            x = find(u + 1)
            while x < v:
                parent[x] = x + 1; alive -= 1
                x = find(x + 1)
        out.append(alive - 1)
    return out

for n, q, exp in examples:
    print(dsu_next_alive(n, q), '| kỳ vọng', exp)

[3, 2, 1] | kỳ vọng [3, 2, 1]
[1, 1] | kỳ vọng [1, 1]


Điều kiện `u` còn sống quan trọng: nếu `u` đã bị một khoảng lớn hơn che phủ thì đường mới (lồng bên trong) không được dùng.

## 8. TRACE từng bước

Trace cách 3 với `n = 8` và các truy vấn lồng/rời nhau.

In [7]:
def trace(n, queries):
    nxt = list(range(1, n + 1)); alive = n
    for u, v in queries:
        removed = []
        if nxt[u] != -1 and nxt[u] < v:
            j = nxt[u]
            while j < v:
                k = nxt[j]; nxt[j] = -1; alive -= 1; removed.append(j); j = k
            nxt[u] = v
            status = f'loại {removed}'
        else:
            status = 'u đã bị che' if nxt[u] == -1 else f'không hữu ích (nxt[{u}] = {nxt[u]} >= {v})'
        path = [0]
        while path[-1] != n - 1:
            path.append(nxt[path[-1]])
        print(f'thêm {u} -> {v}: {status:28s} | đường đi {path} | độ dài {alive - 1}')
    print('BFS:', brute(n, queries))

trace(8, [[2, 5], [3, 5], [1, 6], [6, 7], [2, 4]])

thêm 2 -> 5: loại [3, 4]                  | đường đi [0, 1, 2, 5, 6, 7] | độ dài 5
thêm 3 -> 5: u đã bị che                  | đường đi [0, 1, 2, 5, 6, 7] | độ dài 5
thêm 1 -> 6: loại [2, 5]                  | đường đi [0, 1, 6, 7] | độ dài 3
thêm 6 -> 7: không hữu ích (nxt[6] = 7 >= 7) | đường đi [0, 1, 6, 7] | độ dài 3
thêm 2 -> 4: u đã bị che                  | đường đi [0, 1, 6, 7] | độ dài 3
BFS: [5, 5, 3, 3, 3]


## 9. Code nộp LeetCode

In [8]:
class Solution:
    def shortestDistanceAfterQueries(self, n: int, queries: List[List[int]]) -> List[int]:
        nxt = list(range(1, n + 1))
        alive = n
        ans = []
        for u, v in queries:
            j = nxt[u]
            if j != -1 and j < v:
                while j < v:
                    k = nxt[j]
                    nxt[j] = -1
                    alive -= 1
                    j = k
                nxt[u] = v
            ans.append(alive - 1)
        return ans

for n, q, exp in examples:
    got = Solution().shortestDistanceAfterQueries(n, q)
    print(got, 'OK' if got == exp else 'SAI')

[3, 2, 1] OK
[1, 1] OK


## 10. Vì sao thuật toán đúng

**Bất biến:** các thành phố còn sống `0 = a_0 < a_1 < … < a_k = n−1` tạo thành đường đi `a_0 → a_1 → … → a_k` (mỗi bước là một cạnh có thật), và đây là đường ngắn nhất.

1. **Tính đường đi:** `nxt[a_t] = a_{t+1}` luôn là một cạnh: hoặc `i → i+1` ban đầu, hoặc một đường mới `u → v` đã thêm.
2. **Tối ưu:** mọi cạnh đã thêm `u' → v'` hoặc có `u'` đã chết (nằm trong một khoảng lớn hơn `[u, v]` đang được dùng — nhảy `u → v` không kém), hoặc `[u', v']` nằm trong một bước `[a_t, a_{t+1}]` (không giúp được), hoặc chính nó là một bước. Do không cắt chéo, không cạnh nào "vượt qua" một thành phố còn sống `a_t` để rút ngắn mà không chứa trọn một bước.
3. **Cập nhật:** đường mới `u → v` với `u` sống và `nxt[u] < v`: các thành phố sống trong `(u, v)` chỉ có thể tới được qua đường đi hiện tại, nay nhảy thẳng tốt hơn — loại chúng. Do không cắt chéo, `v` là một thành phố còn sống (nếu `v` đã chết thì có một khoảng chứa `v` ở trong, cắt chéo với `[u, v]` — trừ khi chứa cả `u`, khi đó `u` đã chết).

## 11. Phân tích Time / Space Complexity

| Cách | Time | Space |
|---|---|---|
| 1. BFS mỗi truy vấn | `O(q (n + q))` | `O(n + q)` |
| 2. DP tô-pô mỗi truy vấn | `O(q (n + q))` | `O(n + q)` |
| 3. Danh sách liên kết | `O(n + q)` | `O(n)` |
| 4. DSU kế tiếp còn sống | `O((n + q) α(n))` | `O(n)` |

## 12. Bộ test

### 12.1. Ví dụ đề + ca biên

In [9]:
impls = {'dp_recompute': dp_recompute, 'linked_list': linked_list, 'dsu': dsu_next_alive,
         'submit': lambda n, q: Solution().shortestDistanceAfterQueries(n, q)}
edge = [(3, [[0, 2]], [1]), (6, [[1, 4], [1, 3], [0, 5]], [3, 3, 1]), (6, [[0, 2], [2, 4], [4, 5]], [4, 3, 3]),
        (7, [[1, 5], [0, 1], [5, 6]], None)]
for n, q, exp in examples + edge:
    e = brute(n, q)
    if exp is not None:
        assert e == exp, (n, q, e)
    for name, f in impls.items():
        assert f(n, q) == e, (name, n, q)
print('Ví dụ đề và ca biên: tất cả đúng.')

Ví dụ đề và ca biên: tất cả đúng.


Ca thứ hai: `1 → 3` nằm **trong** `1 → 4` đã có ⇒ không đổi; `0 → 5` bao tất cả ⇒ `1`.

### 12.2. Đối chiếu ngẫu nhiên với BFS (sinh truy vấn không cắt chéo)

In [10]:
import random
rng = random.Random(3244)

def random_noncrossing(n, q):
    """Sinh q khoảng [u, v] (v - u >= 2) đôi một không cắt chéo, theo thứ tự ngẫu nhiên."""
    chosen = []
    tries = 0
    while len(chosen) < q and tries < 2000:
        tries += 1
        u = rng.randrange(n - 2); v = rng.randrange(u + 2, n)
        if (u, v) in chosen:
            continue
        if all(not (a < u < b < v or u < a < v < b) for a, b in chosen):
            chosen.append((u, v))
    return [list(p) for p in chosen]

cnt = 0
for _ in range(500):
    n = rng.randint(3, 15)
    qs = random_noncrossing(n, rng.randint(1, 8))
    if not qs:
        continue
    exp = brute(n, qs)
    for name, f in impls.items():
        assert f(n, qs) == exp, (name, n, qs)
    cnt += 1
print('Khớp BFS trên', cnt, 'bộ test ngẫu nhiên không cắt chéo.')

Khớp BFS trên 500 bộ test ngẫu nhiên không cắt chéo.


### 12.3. Ca lớn `n = q = 10^5`

In [11]:
import time
N = 10 ** 5
# Các khoảng lồng nhau [i, N-1-i] (thêm từ trong ra ngoài) + rời nhau dạng [2k, 2k+2]
nested = [[i, N - 1 - i] for i in range(N // 2 - 2, -1, -1)]
disjoint = [[2 * k, 2 * k + 2] for k in range((N - 3) // 2)]
for name, qs in [('lồng nhau (trong ra ngoài)', nested), ('rời nhau', disjoint), ('rời nhau, đảo thứ tự', disjoint[::-1])]:
    t = time.time(); a = Solution().shortestDistanceAfterQueries(N, qs); t1 = time.time() - t
    t = time.time(); b = dsu_next_alive(N, qs); t2 = time.time() - t
    assert a == b
    print(f"{name:28s}: q = {len(qs):6d}, đáp án cuối {a[-1]:6d} | nộp {t1:.2f}s | DSU {t2:.2f}s")
assert Solution().shortestDistanceAfterQueries(N, nested)[-1] == 1
print('lồng nhau: cuối cùng có đường 0 -> N-1 trực tiếp -> 1.')

lồng nhau (trong ra ngoài)  : q =  49999, đáp án cuối      1 | nộp 0.02s | DSU 0.05s
rời nhau                    : q =  49998, đáp án cuối  50001 | nộp 0.02s | DSU 0.03s
rời nhau, đảo thứ tự        : q =  49998, đáp án cuối  50001 | nộp 0.01s | DSU 0.03s


lồng nhau: cuối cùng có đường 0 -> N-1 trực tiếp -> 1.


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Không kiểm tra `u` đã bị che (nested)

In [12]:
def wrong_no_alive_check(n, queries):
    nxt = list(range(1, n + 1)); alive = n; out = []
    for u, v in queries:
        j = nxt[u]
        if j < v:                                             # SAI: j = -1 (u đã chết) cũng thỏa j < v
            while j < v:
                k = nxt[j]; nxt[j] = -1; alive -= 1; j = k     # nxt[-1] trỏ về phần tử CUỐI!
            nxt[u] = v
        out.append(alive - 1)
    return out

print('sai :', wrong_no_alive_check(4, [[0, 3], [1, 3]]), '| đúng:', Solution().shortestDistanceAfterQueries(4, [[0, 3], [1, 3]]))

sai : [1, 0] | đúng: [1, 1]


**Vì sao sai:** thành phố `1` đã bị khoảng `[0, 3]` loại (`nxt[1] = −1`). Bản sai coi `−1 < 3` là "đường mới vươn xa hơn", rồi `nxt[−1]` của Python trỏ tới phần tử **cuối** mảng — "loại" nhầm thành phố `n − 1`. Phải kiểm tra `u` còn sống trước.

### Lỗi 2 — Trả về số thành phố còn sống (quên trừ 1)

In [13]:
def wrong_count_nodes(n, queries):
    return [x + 1 for x in Solution().shortestDistanceAfterQueries(n, queries)]     # SAI: số đỉnh, không phải số cạnh

print('sai :', wrong_count_nodes(*examples[0][:2]), '| đúng:', Solution().shortestDistanceAfterQueries(*examples[0][:2]))

sai : [4, 3, 2] | đúng: [3, 2, 1]


**Vì sao sai:** độ dài đường đi là số **cạnh** = số đỉnh − 1.

### Lỗi 3 — Cập nhật kiểu `dist −= (v − u − 1)` không theo dõi chồng lấn

In [14]:
def wrong_subtract(n, queries):
    dist = n - 1; out = []
    for u, v in queries:
        dist = min(dist, dist - (v - u - 1))                   # SAI: đếm trùng phần đã rút ngắn
        out.append(dist)
    return out

print('sai :', wrong_subtract(*examples[1][:2]), '| đúng:', Solution().shortestDistanceAfterQueries(*examples[1][:2]))

sai : [1, 0] | đúng: [1, 1]


**Vì sao sai:** đường `0 → 2` nằm trong `0 → 3` đã có — không rút ngắn thêm, nhưng công thức trừ thêm `1`.

### Lỗi 4 — Ghi đè `nxt[u] = v` cả khi `nxt[u] ≥ v`

In [15]:
def wrong_overwrite(n, queries):
    nxt = list(range(1, n + 1)); alive = n; out = []
    for u, v in queries:
        if nxt[u] != -1:
            j = nxt[u]
            while j < v:
                k = nxt[j]; nxt[j] = -1; alive -= 1; j = k
            nxt[u] = v                                         # SAI khi v < nxt[u]: rút ngắn bước nhảy
        out.append(alive - 1)
    return out

bad = (5, [[0, 3], [0, 2], [0, 4]])
print('sai :', wrong_overwrite(*bad), '| đúng:', Solution().shortestDistanceAfterQueries(*bad))

sai : [2, 2, 0] | đúng: [2, 2, 1]


**Vì sao sai:** sau `0 → 3`, truy vấn `0 → 2` vô dụng nhưng bản sai đặt `nxt[0] = 2` (thành phố đã bị loại) — phá vỡ chuỗi. Truy vấn `0 → 4` sau đó đi theo chuỗi hỏng và "loại" lại các thành phố đã chết (thậm chí đọc `nxt[−1]`), làm đáp án sai.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- Điều kiện "không cắt chéo" ⇒ cấu trúc **lồng nhau** (như ngoặc) ⇒ mỗi phần tử bị "nuốt" tối đa một lần ⇒ phân tích khấu trừ `O(n)`.
- "Phần tử còn sống kế tiếp" ⇒ danh sách liên kết hoặc DSU với nén đường đi.
- Luôn kiểm tra trường hợp đường mới nằm **bên trong** một đường cũ.

## 15. Biến thể của bài

1. **LC 3243 (bản I):** cho phép cắt chéo, `n, q ≤ 500` ⇒ BFS/DP mỗi truy vấn.
2. Đường có **trọng số** ⇒ không còn tính chất "nuốt"; cần DP/Dijkstra.
3. Truy vấn **xóa** đường ⇒ khó hơn nhiều (cần cấu trúc lồng nhau như cây khoảng).
4. Hỏi khoảng cách từ `0` tới một thành phố `x` **còn sống** ⇒ số thành phố sống `< x` (Fenwick trên trạng thái sống).

In [16]:
# Biến thể 1: cho phép cắt chéo — DP mỗi truy vấn (cách 2) vẫn đúng; lời giải danh sách liên kết thì không
def crossing_counterexample():
    n, qs = 6, [[2, 5], [0, 3]]                 # 0 < 2 < 3 < 5: cắt chéo
    return n, qs, dp_recompute(n, qs), Solution().shortestDistanceAfterQueries(n, qs), brute(n, qs)

n_, qs_, a_, b_, c_ = crossing_counterexample()
print(f'cắt chéo {qs_}: DP = {a_}, danh sách liên kết = {b_}, BFS = {c_}')
assert a_ == c_ and b_ != c_

cắt chéo [[2, 5], [0, 3]]: DP = [3, 3], danh sách liên kết = [3, 1], BFS = [3, 3]


Với đường cắt chéo, khi thêm `0 → 3` sau `2 → 5`, lời giải danh sách liên kết "loại" cả `1` và `2` rồi nhảy tới `5` — như thể có đường `0 → 5`, cho đáp án `1`, trong khi đường thật ngắn nhất vẫn dài `3` (`0 → 3 → 4 → 5` hoặc `0 → 1 → 2 → 5`). Điều kiện không cắt chéo của đề là then chốt.

## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Ghi chú |
|---|---|---|---|
| 1 | BFS mỗi truy vấn | `O(q(n+q))` | Oracle |
| 2 | DP tô-pô mỗi truy vấn | `O(q(n+q))` | Không cần điều kiện đề |
| 3 | Danh sách liên kết | `O(n + q)` | **Nộp bài** |
| 4 | DSU kế tiếp còn sống | `O((n+q) α)` | Tương đương |

## 17. 5 bài tập vận dụng (từ dễ đến khó)

**Bài 1 (đoán output):** `n = 3`, `[[0, 2]]`.

**Bài 2 (tính tay):** `n = 7`, `[[1, 3], [4, 6], [0, 4]]`.

**Bài 3 (điền TODO):** trả về luôn **đường đi** (danh sách thành phố) sau mỗi truy vấn với `n` nhỏ.

**Bài 4 (tự viết lại):** biến thể 1 — `n, q ≤ 500`, cho phép cắt chéo (LC 3243).

**Bài 5 (mini challenge):** đếm số truy vấn **vô dụng** (không làm thay đổi đáp án) trong `O(n + q)`.

In [17]:
def paths_todo(n, queries):
    # TODO
    raise NotImplementedError

def crossing_allowed_todo(n, queries):
    # TODO
    raise NotImplementedError

def useless_count_todo(n, queries):
    # TODO
    raise NotImplementedError

print('Đã định nghĩa các hàm TODO (không gọi).')

Đã định nghĩa các hàm TODO (không gọi).


## 18. Gợi ý 3 mức cho bài tập

**Bài 1.** (1) `0 → 2` trực tiếp. (2) Loại `1`. (3) `[1]`.

**Bài 2.** (1) `1 → 3` loại `2` ⇒ 5. (2) `4 → 6` loại `5` ⇒ 4. (3) `0 → 4` loại `1, 3` ⇒ còn `0, 4, 6` ⇒ `[5, 4, 2]`. (Chú ý: `[0, 5]` sẽ cắt chéo `[4, 6]` — không hợp lệ theo đề.)

**Bài 3.** (1) Đi theo `nxt` từ 0. (2) Ghi lại danh sách. (3) Độ dài − 1 = đáp án.

**Bài 4.** (1) DP tô-pô cách 2. (2) Hoặc BFS. (3) `O(q (n + q))` đủ cho 500.

**Bài 5.** (1) Truy vấn vô dụng ⇔ không loại thành phố nào. (2) Đếm trong vòng lặp của cách 3. (3) `O(n + q)`.

## 19. Đáp án tham khảo cho bài tập

In [18]:
# Bài 1, 2, 3
assert Solution().shortestDistanceAfterQueries(3, [[0, 2]]) == [1]
assert Solution().shortestDistanceAfterQueries(7, [[1, 3], [4, 6], [0, 4]]) == [5, 4, 2] == brute(7, [[1, 3], [4, 6], [0, 4]])

def paths(n, queries):
    nxt = list(range(1, n + 1)); res = []
    for u, v in queries:
        if nxt[u] != -1 and nxt[u] < v:
            j = nxt[u]
            while j < v:
                k = nxt[j]; nxt[j] = -1; j = k
            nxt[u] = v
        p = [0]
        while p[-1] != n - 1:
            p.append(nxt[p[-1]])
        res.append(p)
    return res

ps = paths(*examples[0][:2])
assert [len(p) - 1 for p in ps] == examples[0][2]
print('Bài 1: [1] | Bài 2: [5, 4, 2] | Bài 3:', ps)

Bài 1: [1] | Bài 2: [5, 4, 2] | Bài 3: [[0, 1, 2, 4], [0, 2, 4], [0, 4]]


In [19]:
# Bài 4, 5
def crossing_allowed(n, queries):
    return dp_recompute(n, queries)

def useless_count(n, queries):
    nxt = list(range(1, n + 1)); useless = 0
    for u, v in queries:
        if nxt[u] != -1 and nxt[u] < v:
            j = nxt[u]
            while j < v:
                k = nxt[j]; nxt[j] = -1; j = k
            nxt[u] = v
        else:
            useless += 1
    return useless

rng = random.Random(4)
for _ in range(200):
    n = rng.randint(3, 12)
    qs = [[u, rng.randrange(u + 2, n)] for u in [rng.randrange(n - 2) for _ in range(rng.randint(1, 6))]]
    qs = [list(t) for t in dict.fromkeys(map(tuple, qs))]
    assert crossing_allowed(n, qs) == brute(n, qs)
    qn = random_noncrossing(n, rng.randint(1, 6))
    if qn:
        res = Solution().shortestDistanceAfterQueries(n, qn)
        prev = [n - 1] + res[:-1]
        assert useless_count(n, qn) == sum(1 for a, b in zip(prev, res) if a == b)
print('Bài 4 khớp BFS (kể cả cắt chéo); Bài 5: truy vấn vô dụng = truy vấn không đổi đáp án.')

Bài 4 khớp BFS (kể cả cắt chéo); Bài 5: truy vấn vô dụng = truy vấn không đổi đáp án.


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

```python
nxt = [i + 1 for i in range(n)]; alive = n
for u, v in queries:
    j = nxt[u]
    if j != -1 and j < v:
        while j < v: k = nxt[j]; nxt[j] = -1; alive -= 1; j = k
        nxt[u] = v
    ans.append(alive - 1)
```

**Checklist:**

- [ ] Kiểm tra `u` còn sống (`nxt[u] != −1`)?
- [ ] Chỉ cập nhật khi `nxt[u] < v`?
- [ ] Đáp án = số thành phố còn sống − 1?
- [ ] Hiểu vì sao cần điều kiện không cắt chéo?
- [ ] Mỗi thành phố bị loại tối đa một lần (`O(n + q)`)?